# HydroSeason: End-to-End Water-Extent Workflow

This notebook is the runnable example for the current HydroSeason workflow.
It starts from monthly surface-water extent, classifies the seasonal structure,
selects suitable analyses, detects hydrological years, labels individual months,
runs condition/anomaly workflows, visualizes results, and writes an interactive
HTML report.

Default input source is STAC/WOfS monthly water masks. A CSV monthly-extent
path is included as a deterministic offline option and fallback; downstream
analysis is identical because both paths converge on `extent_pct` / `invalid_pct`.

Terminology note: HydroSeason reports surface-water extent timing and condition.
The stress screen below is a surface-water extent screening table, not discharge,
volume, drought, ecological condition, or causal attribution.

## 1. Imports

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:  # matplotlib is optional for the core CSV/report workflow
    plt = None

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "hydroseason").is_dir() and (candidate / "notebooks").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydroseason import (
    analyze_hydrological_state,
    classify_seasonal_pattern,
    detect_hydrological_years,
    generate_html_report,
    label_hydrological_months,
    suggest_hydro_year_config,
)
from hydroseason.examples import (
    dynamic_hydro_years_for_report,
    flag_extent_quality,
    load_workflow_extent,
)

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

print("Imports successful")

## 2. Setup

In [ ]:
# INPUT_SOURCE selects the preferred acquisition path. STAC is the default
# production workflow; use "csv" when you already have monthly extent.
INPUT_SOURCE = "stac"

# REPORT_PATH is the self-contained HTML summary written by the final step.
REPORT_PATH = NOTEBOOK_DIR / "hydroseason_analysis_report.html"

# CATCHMENT_KEY selects the real catchment CSV used when CSV is selected or
# when the offline STAC fallback is active.
CATCHMENT_KEY = "fitzroy_river_wa"

# RESOLUTION selects the precomputed catchment sample-area resolution.
RESOLUTION = "30m"

# RUN_REMOTE_STAC controls whether this notebook actually queries the remote
# STAC catalog. Keep False for offline documentation/tests; set True for a real run.
RUN_REMOTE_STAC = False

# CSV_FALLBACK_WHEN_STAC_DISABLED keeps the default STAC notebook runnable when
# RUN_REMOTE_STAC is False or optional STAC dependencies are unavailable.
CSV_FALLBACK_WHEN_STAC_DISABLED = True

# STAC_URL is the catalog endpoint used by the default WOfS acquisition path.
STAC_URL = "https://explorer.dea.ga.gov.au/stac"

# COLLECTION is the DEA Landsat WOfS product queried through STAC.
COLLECTION = "ga_ls_wo_3"

# AOI_PATH points to the polygon used to clip raster/STAC water masks.
AOI_PATH = PROJECT_ROOT / "data" / "fitzroy_kimberley_aoi.geojson"

# OUTPUT_CRS is the projected CRS used for WOfS loading and AOI clipping.
OUTPUT_CRS = 3577

# START_DATE and END_DATE bound the monthly water-extent analysis period.
START_DATE = "2006-01-01"
END_DATE = "2025-12-31"

# CSV_DATA_ROOT is the directory containing the existing catchment CSVs.
CSV_DATA_ROOT = PROJECT_ROOT / "output" / "resolution_window_comparison"
# DATA_PATH is derived from the selected catchment and resolution.
DATA_PATH = CSV_DATA_ROOT / CATCHMENT_KEY / f"extent_{RESOLUTION}.csv"

# TIME_BLOCK bounds monthly STAC extent computation by reducing this many time steps per chunk.
TIME_BLOCK = 12

# MAX_INVALID_PCT is the conservative invalid-coverage threshold. Months
# above it are flagged as potentially biased; they do not
# block the example workflow.
MAX_INVALID_PCT = 10

# Rolling condition baselines need enough prior cycles, then use a trailing window.
ROLLING_MIN_CYCLES = 5
ROLLING_WINDOW_CYCLES = 10

print(f"Input source preference: {INPUT_SOURCE}")
print(f"Remote STAC execution enabled: {RUN_REMOTE_STAC}")
print(f"Analysis period: {START_DATE} to {END_DATE}")
print(f"Catchment CSV: {DATA_PATH}")
print(f"Bias threshold: invalid_pct <= {MAX_INVALID_PCT}% for quality flagging")

## 3. Get Water Extent

STAC/WOfS is the default acquisition path: load an AOI, query the STAC catalog,
clip monthly water masks, then summarize to monthly extent. The CSV path is an
option for precomputed monthly extent and for offline example execution.

In [ ]:
raw_extent, extent_source = load_workflow_extent(
    input_source=INPUT_SOURCE,
    run_remote_stac=RUN_REMOTE_STAC,
    csv_fallback_when_stac_disabled=CSV_FALLBACK_WHEN_STAC_DISABLED,
    csv_path=DATA_PATH,
    stac_url=STAC_URL,
    collection=COLLECTION,
    aoi_path=AOI_PATH,
    start_date=START_DATE,
    end_date=END_DATE,
    periods=240,
    crs=OUTPUT_CRS,
    cache_dir=PROJECT_ROOT / "output" / "example_extent_cache",
    time_block=TIME_BLOCK,
)

print(f"Extent source used: {extent_source}")
print(f"Loaded monthly extent: {raw_extent.shape[0]} months")
print(f"Observed data range: {raw_extent.index.min().date()} to {raw_extent.index.max().date()}")
print(f"Observed data path: {DATA_PATH}")
print(f"Raw max invalid_pct: {raw_extent['invalid_pct'].max():.1f}%")

extent, quality = flag_extent_quality(raw_extent, max_invalid_pct=MAX_INVALID_PCT)
quality_summary = pd.DataFrame([
    {
        "months": len(quality),
        "flagged_months": int(quality['quality_flag'].ne('usable').sum()),
        "high_invalid_months": int(quality['quality_flag'].eq('high_invalid_pct').sum()),
        "missing_extent_months": int(quality['quality_flag'].eq('missing_extent').sum()),
        "max_invalid_pct": float(quality['invalid_pct'].max()),
        "mean_invalid_pct": float(quality['invalid_pct'].mean()),
        "bias_warning": "Potential bias flagged; no values were filled.",
    }
])
quality_summary

### CSV Option

Set `INPUT_SOURCE = "csv"` to use the selected precomputed catchment extent
directly. Change `CATCHMENT_KEY` and `RESOLUTION` to select another CSV in
`output/resolution_window_comparison`; the same path is used by the offline
STAC fallback.

In [ ]:
# The loader above now uses the selected real catchment CSV. Inspect raw
# quality before classification; high invalid coverage is retained as a
# bias warning. The values in `extent` remain observed; no gap filling occurs.
quality.loc[quality['quality_flag'] != "usable", ["extent_pct_raw", "invalid_pct", "quality_flag", "bias_warning"]].head(12)

## 4. Classify Data Structure

`classify_seasonal_pattern` decides whether the monthly extent record is
unimodal annual, bimodal/complex, weak/irregular, low variability, or too short.
That result determines which analyses are scientifically safe to run.

In [ ]:
pattern = classify_seasonal_pattern(extent, n_bootstrap=80, random_state=0, quality_policy="flag")

pattern_summary = pd.DataFrame(
    [
        {
            "pattern": pattern.pattern,
            "expected_peak_month": pattern.expected_peak_month,
            "expected_trough_month": pattern.expected_trough_month,
            "secondary_peak_month": pattern.secondary_peak_month,
            "seasonal_strength": round(pattern.seasonal_strength, 3),
            "bootstrap_support": round(pattern.bootstrap_support, 3),
            "complete_years": pattern.n_complete_years,
        }
    ]
)
pattern_summary

## 5. Decide What Can Run

This gate makes the workflow explicit before detection starts. A stable
unimodal annual signal is the strongest case for dynamic hydrological-year
detection. Bimodal/complex systems need reviewed fixed windows or expert rules
before month labels/report summaries are interpreted.

In [ ]:
def workflow_gate(pattern_name: str) -> pd.DataFrame:
    has_enough_baseline = pattern.n_complete_years >= ROLLING_MIN_CYCLES
    rows = [
        {
            "workflow": "Dynamic hydrological-year detection + annual condition",
            "status": "run" if pattern_name == "unimodal_annual" else "needs_expert_trough",
            "why": "Best fit for one stable annual peak and one stable trough.",
        },
        {
            "workflow": "Fixed-window hydrological year detection + month labels",
            "status": "reviewed_windows" if pattern_name == "bimodal_or_complex" else "optional_report_path",
            "why": "Useful when complex seasonality needs explicit, reviewed wet/dry windows; also feeds the current HTML report.",
        },
        {
            "workflow": "Monthly same-calendar-month anomaly",
            "status": "run" if has_enough_baseline else "needs_longer_record",
            "why": "Needs enough same-month baseline observations.",
        },
        {
            "workflow": "Surface-water stress screen",
            "status": "screen_only" if has_enough_baseline else "needs_condition_baseline",
            "why": "Use dry/refuge condition labels as extent-condition screening, not causal drought/stress attribution.",
        },
        {
            "workflow": "Interactive HTML report",
            "status": "run_with_fixed_window_result",
            "why": "Report currently summarizes the fixed-window hydrological-year result.",
        },
    ]
    return pd.DataFrame(rows)

workflow_decision = workflow_gate(pattern.pattern)
workflow_decision

## 6. Run Hydrological Year Detection and Label Individual Months

For `unimodal_annual`, dynamic hydrological-year detection is the primary
result and labels individual months. For `bimodal_or_complex`, the reviewed
fixed-window result is primary. The fixed-window result is also retained as
the report-compatible summary path.

In [ ]:
hydro_config = suggest_hydro_year_config(extent)
print(f"Suggested fixed-window config: {hydro_config}")

hydro_years = detect_hydrological_years(
    extent,
    config=hydro_config,
    max_invalid_pct=MAX_INVALID_PCT,
    quality_policy="flag",
    missing_month_policy="ignore",
)

labels = label_hydrological_months(extent.index, hydro_years)

print(f"Detected fixed-window hydrological years: {len(hydro_years)}")

if pattern.pattern == "unimodal_annual":
    try:
        state = analyze_hydrological_state(
            extent,
            reference="rolling",
            rolling_min_cycles=ROLLING_MIN_CYCLES,
            rolling_window_cycles=ROLLING_WINDOW_CYCLES,
            n_bootstrap=80,
            random_state=0,
            quality_policy="flag",
        )
        state_status = "run"
    except ValueError as exc:
        state = None
        state_status = f"review_first: {exc}"
else:
    state = None
    state_status = f"not selected for pattern={pattern.pattern}; use reviewed fixed windows"

primary_hydro_years = state.hydro_years if state is not None else hydro_years
primary_labels = label_hydrological_months(extent.index, primary_hydro_years)
report_hydro_years = (
    dynamic_hydro_years_for_report(state.hydro_years) if state is not None else hydro_years
)
report_labels = label_hydrological_months(extent.index, report_hydro_years)
print(f"Dynamic hydrological-year workflow: {state_status}")
hydro_years.tail()

In [ ]:
labeled_extent = extent.join(primary_labels)
labeled_extent.tail(12)

## 7. Run Additional Workflows

`analyze_hydrological_state` runs the newer dynamic cycle detection plus annual
recharge/refuge condition and monthly anomaly tables. The rolling baseline below
uses at least five prior complete cycles, then a trailing ten-cycle baseline.

In [ ]:
print(f"Dynamic hydrological-year workflow: {state_status}")

annual_state_cols = [
    "hy_year",
    "status",
    "peak_month",
    "peak_extent_pct",
    "trough_month",
    "trough_extent_pct",
    "annual_condition",
    "annual_condition_qualified",
    "timing_confidence",
]

(state.hydro_years[annual_state_cols].tail(12) if state is not None else pd.DataFrame({"status": [state_status]}))

In [ ]:
if state is None:
    monthly_anomaly = pd.DataFrame({"status": ["Not run: no stable dynamic trough after quality-aware flagging."]})
else:
    monthly_anomaly = state.monthly_condition.join(extent[["extent_pct"]], rsuffix="_input")
    monthly_anomaly = monthly_anomaly[["extent_pct", "reference_median_pct", "anomaly_pct", "condition_percentile", "quality_state"]]
monthly_anomaly.tail(12)

In [ ]:
if state is None:
    surface_water_stress_screen = pd.DataFrame({"status": ["Not run: dynamic hydrological years were unavailable."]})
else:
    surface_water_stress_screen = state.hydro_years.loc[
        state.hydro_years["annual_condition_qualified"].isin(
            ["dry_low_refuge", "buffered_low_recharge"]
        ),
        [
            "hy_year",
            "trough_month",
            "trough_extent_pct",
            "trough_percentile",
            "annual_condition_qualified",
            "consecutive_dry_cycles",
            "timing_confidence",
        ],
    ]

surface_water_stress_screen

## 8. Visualization

In [ ]:
if plt is None:
    print("matplotlib is not installed; skipping inline plot. HTML report generation still works.")
else:
    fig, ax = plt.subplots(figsize=(15, 6))

    for ts, row in report_labels.iterrows():
        color = "#dbeafe" if row["season"] == "Wet" else "#fef3c7"
        ax.axvspan(ts - pd.Timedelta(days=15), ts + pd.Timedelta(days=15), color=color, alpha=0.45, linewidth=0)

    ax.plot(raw_extent.index, raw_extent["extent_pct"], color="#1d4ed8", marker="o", markersize=3, label=f"{CATCHMENT_KEY} observed extent (%)")
    flagged = quality.loc[quality["quality_flag"] != "usable"].dropna(subset=["extent_pct_raw"])
    if not flagged.empty:
        ax.scatter(flagged.index, flagged["extent_pct_raw"], color="#b91c1c", marker="x", s=45, label="Quality-flagged observation", zorder=4)

    for _, row in report_hydro_years.iterrows():
        ax.plot(pd.Timestamp(row["peak_month"]), row["peak_extent_pct"], marker="o", color="#059669", markersize=7)
        ax.plot(pd.Timestamp(row["end_dry_month"]), row["end_extent_pct"], marker="o", color="#dc2626", markersize=7)

    stress_years = set(surface_water_stress_screen["hy_year"]) if "hy_year" in surface_water_stress_screen else set()
    if state is not None:
        for _, row in state.hydro_years[state.hydro_years["hy_year"].isin(stress_years)].iterrows():
            if pd.notna(row["trough_month"]):
                ax.axvline(pd.Timestamp(row["trough_month"]), color="#7c2d12", linestyle="--", linewidth=1, alpha=0.7)

    ax.set_title(f"{CATCHMENT_KEY}: observed monthly extent and selected HY boundaries")
    ax.set_ylabel("Water extent (%)")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(loc="upper right")
    fig.tight_layout()
    plt.show()

## 9. Generate Interactive HTML Summary Report

The report is self-contained HTML and uses the selected primary HY result:
dynamic cycles for unimodal data, reviewed fixed windows for complex data.

In [ ]:
print(f"Report source: {extent_source} | {DATA_PATH}")
print(f"Report extent rows: {len(raw_extent)} | {raw_extent.index.min().date()} to {raw_extent.index.max().date()}")
report_path = generate_html_report(
    extent=raw_extent,
    hydro_years=report_hydro_years,
    output_path=REPORT_PATH,
    title=(
        f"HydroSeason {CATCHMENT_KEY}: {int(quality['quality_flag'].ne('usable').sum())} flagged months; "
        f"invalid coverage above {MAX_INVALID_PCT}% may bias results; no fill applied"
    ),
)

print(f"HTML report written to: {report_path}")